# Small grid droplet simulation

This notebook builds a tiny grid, runs a droplet-style wave simulation, and prepares sparse edge bins plus a static-graph aggregate for later rollout/visualization work.

In [8]:
import sys
from pathlib import Path

root = Path.cwd().resolve()
if root.name == 'notebooks':
    root = root.parent
if (root / 'interactionfields').exists():
    sys.path.insert(0, str(root))

import numpy as np
import scipy.sparse as sp

from interactionfields.graphs import build_graph
from interactionfields.simulate import simulate_faucet_on_graph
from interactionfields.baseline_static import StaticGraphSpec, build_static_graph_from_train


In [9]:
# Small 6x6 grid
A, meta = build_graph('grid', m=6, n=6)
A = A.tocsr()
m, n = meta['shape']
center = (m // 2) * n + (n // 2)

A.shape, center


((36, 36), 21)

In [10]:
# Faucet-driven outward edges
T = 60
bins, H, meta_list = simulate_faucet_on_graph(
    adj=A,
    t_bins=T,
    center_idx=center,
    faucet_period=10,
    speed_hops_per_step=1.0,
    sigma_hops=1.0,
    p_max=0.9,
    return_states=True,
    seed=0,
)

len(bins), H.shape


(60, (60, 36))

In [11]:
# Quick sanity checks on outputs
len(bins), H.shape, len(meta_list)

meta_list[0]


{}

In [12]:
# Aggregate a static graph from the first part of the rollout
T_train = 40
y_train = bins[:T_train]

spec = StaticGraphSpec(window_bins=None, weight_mode='count', undirected=True)
A_static = build_static_graph_from_train(y_train, num_nodes=A.shape[0], spec=spec)

A.nnz, A_static.nnz


(120, 120)

In [13]:
from matplotlib import animation
from IPython.display import HTML
import numpy as np
import matplotlib.pyplot as plt

coords = meta["coords2d"]

# prepare figure
fig, ax = plt.subplots(figsize=(5, 5))
ax.set_aspect("equal")
ax.set_xticks([])
ax.set_yticks([])
pad = 0.5
ax.set_xlim(coords[:, 0].min() - pad, coords[:, 0].max() + pad)
ax.set_ylim(coords[:, 1].min() - pad, coords[:, 1].max() + pad)

nodes_sc = ax.scatter(coords[:, 0], coords[:, 1], c="k", s=40, zorder=3)

# Precompute edge data per bin
frames = []
for B in bins:
    coo = B.tocoo()
    mask = coo.row < coo.col
    rows, cols = coo.row[mask], coo.col[mask]
    frames.append((rows, cols))

lines = []

def update(frame_idx):
    for ln in lines:
        ln.remove()
    lines.clear()
    rows, cols = frames[frame_idx]
    for u, v in zip(rows, cols):
        x = (coords[u, 0], coords[v, 0])
        y = (coords[u, 1], coords[v, 1])
        ln, = ax.plot(x, y, color="#ff6f00", linewidth=2.0, alpha=0.9, zorder=1)
        lines.append(ln)
    ax.set_title(f"Edge events — frame {frame_idx}", fontsize=10)
    return lines

anim = animation.FuncAnimation(fig, update, frames=len(frames), interval=100, blit=False, repeat=True)

# display inline
HTML(anim.to_jshtml())
